# Careem Food: AI Decision Brief Generator
### Optional AI Challenge Submission | Product Manager / Growth Manager Application
**Candidate:** Meldi Hafizh Sayoko  
**Focus:** Careem Food (Dubai / UAE) Telemetry & Funnel Synthesis Engine

---

## 100-Word Summary of Idea & Approach
> Careem Food Decision Brief Generator converts hourly multi-zone delivery telemetry into executive-ready business briefs. The pipeline ingests funnel data, flags statistical outliers across kitchen prep latency, driver supply, and basket sizes, and prompts an LLM with strict product management guardrails. Rather than generating descriptive observations, the system isolates root causes across operational friction and pricing mechanics. It then delivers three prioritized, functional levers: dynamic delivery radius throttling for congested kitchens, minimum order value guardrails on peak vouchers to prevent margin leakage, and in-app scheduled delivery incentives to smooth peak demand while protecting unit economics.

## Step 1: Ingest Telemetry Data
We load synthetic hourly operational telemetry for Dubai delivery zones (Downtown, Marina, Business Bay, JLT, Deira).

In [ ]:
import pandas as pd
import numpy as np

# Direct dataset url or local generation
data_url = 'https://raw.githubusercontent.com/mechateam/careem-decision-brief/main/careem_food_dubai_hourly.csv'
try:
    df = pd.read_csv(data_url)
    print(f'Loaded dataset from GitHub: {df.shape[0]} rows')
except Exception as e:
    print('Using local synthetic data generator fallback...')
    # Fallback to local csv if offline
    df = pd.read_csv('careem_food_dubai_hourly.csv')

df.head()

## Step 2: Funnel Calculation & Anomaly Detection
Calculate end-to-end conversion rates and identify peak-hour operational anomalies.

In [ ]:
# Filter for Downtown Dubai Peak Incident vs Baseline
target_zone = 'Downtown Dubai'
incident_mask = (df['zone'] == target_zone) & (df['timestamp'].str.contains('2026-09-04 19:|2026-09-04 20:|2026-09-04 21:'))
baseline_mask = (df['zone'] == target_zone) & (~incident_mask)

target = df[incident_mask]
baseline = df[baseline_mask]

target_cvr = (target['orders_completed'].sum() / target['sessions'].sum()) * 100
baseline_cvr = (baseline['orders_completed'].sum() / baseline['sessions'].sum()) * 100
target_prep = target['kitchen_prep_time_min'].mean()
baseline_prep = baseline['kitchen_prep_time_min'].mean()

print(f'Incident Overall CVR: {target_cvr:.2f}% (vs Baseline: {baseline_cvr:.2f}%)')
print(f'Kitchen Prep Delay: {target_prep:.1f} mins (vs Baseline: {baseline_prep:.1f} mins)')

## Step 3: LLM Prompt Synthesis Engine
Format telemetry into a strict PM Decision Brief prompt.

In [ ]:
prompt = f'''You are the Growth Product Manager for Careem Food UAE.
Synthesize the following telemetry into an Executive Decision Brief:
- Incident Window: Friday Peak (19:00 - 22:00) in {target_zone}
- Funnel Conversion: {target_cvr:.2f}% (Baseline: {baseline_cvr:.2f}%)
- Kitchen Prep Time: {target_prep:.1f} min (Baseline: {baseline_prep:.1f} min)
- Driver Supply Gap: {target['driver_supply_gap_pct'].mean():.1f}% (Baseline: {baseline['driver_supply_gap_pct'].mean():.1f}%)
- Net Take Rate: {target['net_take_rate_pct'].mean():.2f}% (Baseline: {baseline['net_take_rate_pct'].mean():.2f}%)

Provide:
1. Executive Summary
2. Root Cause Attribution (Ops vs Pricing vs Supply)
3. 3 Prioritized Business Actions (Operational, Commercial, Product)
4. Guardrail Metrics & SLA targets
'''
print(prompt)